In [1]:
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# ==========================================
# 1. LOAD BOTH TRAIN AND VALIDATION DATASETS
# ==========================================
train_data = np.load("../vocalset_10_3_train_embeddings_gendered.npz")
X_train = train_data['X']
y_train = train_data['y']

val_data = np.load("../vocalset_10_3_val_embeddings_gendered.npz")
X_val = val_data['X']
y_val = val_data['y']

print(f"Training data shape: {X_train.shape}")
print(f"Validation data shape: {X_val.shape}\n")

# ==========================================
# 2. INITIALIZE THE CLASSIC ML MODELS
# ==========================================
models = {
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5),
    "Random Forest (RF)": RandomForestClassifier(n_estimators=100, random_state=42),
    "Support Vector Machine (SVM)": SVC(kernel='linear', random_state=42),
    "Multi-Layer Perceptron (MLP)": MLPClassifier(max_iter=300, random_state=42)
}

# ==========================================
# 3. TRAIN, PREDICT, AND EVALUATE EACH ONE
# ==========================================
baseline_results = {}

for name, model in models.items():
    print(f"Training {name}...")
    
    # Train the brain
    model.fit(X_train, y_train)
    
    # Test on unseen validation data
    predictions = model.predict(X_val)
    
    # Calculate accuracy score
    accuracy = accuracy_score(y_val, predictions)
    baseline_results[name] = accuracy
    
    print(f"-> {name} Baseline Accuracy: {accuracy * 100:.2f}%\n")

# ==========================================
# 4. FINAL BENCHMARK SUMMARY
# ==========================================
print("--- FINAL BASELINE BENCHMARKS ---")
for name, score in baseline_results.items():
    print(f"{name}: {score * 100:.2f}%")

Training data shape: (4569, 1024)
Validation data shape: (1700, 1024)

Training K-Nearest Neighbors (KNN)...
-> K-Nearest Neighbors (KNN) Baseline Accuracy: 67.47%

Training Random Forest (RF)...
-> Random Forest (RF) Baseline Accuracy: 69.29%

Training Support Vector Machine (SVM)...
-> Support Vector Machine (SVM) Baseline Accuracy: 66.82%

Training Multi-Layer Perceptron (MLP)...
-> Multi-Layer Perceptron (MLP) Baseline Accuracy: 70.18%

--- FINAL BASELINE BENCHMARKS ---
K-Nearest Neighbors (KNN): 67.47%
Random Forest (RF): 69.29%
Support Vector Machine (SVM): 66.82%
Multi-Layer Perceptron (MLP): 70.18%


In [2]:
from sklearn.feature_selection import SelectFromModel

print("1. Asking Random Forest to grade the features...")
# We use the Random Forest you already trained to grade the columns
rf_grader = RandomForestClassifier(n_estimators=100, random_state=42)
rf_grader.fit(X_train, y_train)

# This automatically deletes any column that scores below the average importance
selector = SelectFromModel(rf_grader, prefit=True)

# Transform both datasets to drop the useless columns
X_train_reduced = selector.transform(X_train)
X_val_reduced = selector.transform(X_val)

print(f"Original number of features: {X_train.shape[1]}")
print(f"New number of features after cutting the fat: {X_train_reduced.shape[1]}\n")

print("2. Training the MLP on the lightweight dataset...")
mlp_reduced = MLPClassifier(max_iter=300, random_state=42)
mlp_reduced.fit(X_train_reduced, y_train)

# Calculate the new accuracy
new_predictions = mlp_reduced.predict(X_val_reduced)
new_accuracy = accuracy_score(y_val, new_predictions)

print(f"-> New MLP Accuracy: {new_accuracy * 100:.2f}%")

1. Asking Random Forest to grade the features...
Original number of features: 1024
New number of features after cutting the fat: 288

2. Training the MLP on the lightweight dataset...
-> New MLP Accuracy: 69.94%


In [4]:
import numpy as np
from sklearn.neural_network import MLPClassifier

# 1. Get the scores from your already-trained Random Forest
importances = rf_grader.feature_importances_

# Sort the columns by importance (highest to lowest)
# This gives us a ranked list of the column numbers
ranked_indices = np.argsort(importances)[::-1]

# 2. Define the extreme milestones we want to test
feature_targets = [200, 100, 50, 20, 10]

print("--- HUNTING FOR THE SWEET SPOT ---")
print(f"Original Baseline (1024): 70.18%")
print(f"Previous Cut (288): 69.94%\n")

for k in feature_targets:
    # Grab only the top 'k' column indices
    top_k_cols = ranked_indices[:k]
    
    # Slice both datasets to keep only those specific columns
    X_train_k = X_train[:, top_k_cols]
    X_val_k = X_val[:, top_k_cols]
    
    # Train a fresh MLP on this tiny dataset
    mlp_k = MLPClassifier(max_iter=500, random_state=42)
    mlp_k.fit(X_train_k, y_train)
    
    # Score it
    acc_k = mlp_k.score(X_val_k, y_val)
    
    print(f"Top {k} Features -> MLP Accuracy: {acc_k * 100:.2f}%")

--- HUNTING FOR THE SWEET SPOT ---
Original Baseline (1024): 70.18%
Previous Cut (288): 69.94%

Top 200 Features -> MLP Accuracy: 71.47%
Top 100 Features -> MLP Accuracy: 69.65%


c:\Users\Ivanh\upf-internship\upf-audio\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Top 50 Features -> MLP Accuracy: 67.47%


c:\Users\Ivanh\upf-internship\upf-audio\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Top 20 Features -> MLP Accuracy: 61.12%
Top 10 Features -> MLP Accuracy: 58.94%


c:\Users\Ivanh\upf-internship\upf-audio\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier

# ==========================================
# 0. LOAD DATA 
# ==========================================
train_data = np.load("../vocalset_10_3_train_embeddings_gendered.npz")
X_train = train_data['X']
y_train = train_data['y']

val_data = np.load("../vocalset_10_3_val_embeddings_gendered.npz")
X_val = val_data['X']
y_val = val_data['y']

# We will test a few different "blended" sizes to map out a new curve
pca_targets = [200, 100, 50, 20]

print("--- HUNTING FOR THE SWEET SPOT (USING PCA) ---")
print("Original Baseline (1024 features): 70.18%")
print("Best Random Forest Cut (200 features): 71.47%\n")

for components in pca_targets:
    # 1. Initialize the PCA "blender" to the target size
    pca = PCA(n_components=components, random_state=42)
    
    # 2. Fit the blender on the training data, and transform both sets
    # We use fit_transform on train, but ONLY transform on val (to prevent cheating)
    X_train_pca = pca.fit_transform(X_train)
    X_val_pca = pca.transform(X_val)
    
    # 3. Train a fresh MLP on the blended data
    mlp_pca = MLPClassifier(max_iter=500, random_state=42)
    mlp_pca.fit(X_train_pca, y_train)
    
    # 4. Score it
    acc_pca = mlp_pca.score(X_val_pca, y_val)
    
    print(f"PCA ({components} Components) -> MLP Accuracy: {acc_pca * 100:.2f}%")

--- HUNTING FOR THE SWEET SPOT (USING PCA) ---
Original Baseline (1024 features): 70.18%
Best Random Forest Cut (200 features): 71.47%

PCA (200 Components) -> MLP Accuracy: 69.82%
PCA (100 Components) -> MLP Accuracy: 71.29%
PCA (50 Components) -> MLP Accuracy: 69.47%
PCA (20 Components) -> MLP Accuracy: 68.00%


c:\Users\Ivanh\upf-internship\upf-audio\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import pandas as pd

# ==========================================
# LOAD DATA AND GET RF FEATURE IMPORTANCES
# ==========================================
train_data = np.load("../vocalset_10_3_train_embeddings_gendered.npz")
X_train = train_data['X']
y_train = train_data['y']

val_data = np.load("../vocalset_10_3_val_embeddings_gendered.npz")
X_val = val_data['X']
y_val = val_data['y']

# Train Random Forest to get feature importance scores
print("Computing feature importances...")
rf_importance = RandomForestClassifier(n_estimators=100, random_state=42)
rf_importance.fit(X_train, y_train)
importances = rf_importance.feature_importances_

# Rank features by importance (highest to lowest)
ranked_indices = np.argsort(importances)[::-1]

# ==========================================
# TEST EACH ALGORITHM WITH DIFFERENT FEATURE COUNTS
# ==========================================
feature_counts = [20, 50, 100, 200, 400, 600, 800, 1024]

algorithms = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='linear', random_state=42),
    "MLP": MLPClassifier(max_iter=500, random_state=42)
}

# Store results in a dictionary
results = {algo: [] for algo in algorithms.keys()}

print("\n" + "="*70)
print("FEATURE REDUCTION COMPARISON ACROSS ALL ALGORITHMS")
print("="*70 + "\n")

for num_features in feature_counts:
    # Select top-k features
    top_k_cols = ranked_indices[:num_features]
    X_train_reduced = X_train[:, top_k_cols]
    X_val_reduced = X_val[:, top_k_cols]
    
    print(f"Testing with {num_features:4d} features...")
    
    for algo_name, algo_model in algorithms.items():
        # Train algorithm on reduced features
        algo_model.fit(X_train_reduced, y_train)
        accuracy = algo_model.score(X_val_reduced, y_val)
        
        results[algo_name].append({
            'features': num_features,
            'accuracy': accuracy * 100
        })
        
        print(f"  -> {algo_name:15s}: {accuracy*100:6.2f}%")
    print()

# ==========================================
# CREATE COMPARISON TABLE
# ==========================================
print("\n" + "="*70)
print("SUMMARY: BEST ACCURACY FOR EACH FEATURE COUNT")
print("="*70 + "\n")

comparison_data = []
for num_features in feature_counts:
    row = {'Features': num_features}
    for algo_name in algorithms.keys():
        acc = next(r['accuracy'] for r in results[algo_name] if r['features'] == num_features)
        row[algo_name] = f"{acc:.2f}%"
    comparison_data.append(row)

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

# ==========================================
# IDENTIFY OPTIMAL POINT FOR EACH ALGORITHM
# ==========================================
print("\n" + "="*70)
print("OPTIMAL CONFIGURATION PER ALGORITHM (Highest Accuracy)")
print("="*70 + "\n")

for algo_name in algorithms.keys():
    best_result = max(results[algo_name], key=lambda x: x['accuracy'])
    print(f"{algo_name:15s}: {best_result['accuracy']:6.2f}% accuracy with {best_result['features']:4d} features")

# ==========================================
# EFFICIENCY RANKINGS
# ==========================================
print("\n" + "="*70)
print("EFFICIENCY ANALYSIS: Fewest Features for ≥70% Accuracy")
print("="*70 + "\n")

efficiency = []
for algo_name in algorithms.keys():
    # Find minimum features needed to reach 70% accuracy
    efficient_results = [r for r in results[algo_name] if r['accuracy'] >= 70.0]
    if efficient_results:
        min_features = min(efficient_results, key=lambda x: x['features'])
        efficiency.append({
            'Algorithm': algo_name,
            'Min Features': min_features['features'],
            'Accuracy': f"{min_features['accuracy']:.2f}%"
        })
    else:
        efficiency.append({
            'Algorithm': algo_name,
            'Min Features': 'N/A',
            'Accuracy': 'Cannot reach 70%'
        })

df_efficiency = pd.DataFrame(efficiency)
print(df_efficiency.to_string(index=False))


Computing feature importances...

FEATURE REDUCTION COMPARISON ACROSS ALL ALGORITHMS

Testing with   20 features...
  -> KNN            :  62.12%
  -> Random Forest  :  65.53%
  -> SVM            :  62.82%


c:\Users\Ivanh\upf-internship\upf-audio\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


  -> MLP            :  61.12%

Testing with   50 features...
  -> KNN            :  63.12%
  -> Random Forest  :  67.00%
  -> SVM            :  68.00%


c:\Users\Ivanh\upf-internship\upf-audio\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


  -> MLP            :  67.47%

Testing with  100 features...
  -> KNN            :  66.00%
  -> Random Forest  :  67.65%
  -> SVM            :  67.47%
  -> MLP            :  69.65%

Testing with  200 features...
  -> KNN            :  67.94%
  -> Random Forest  :  68.94%
  -> SVM            :  68.41%
  -> MLP            :  71.47%

Testing with  400 features...
  -> KNN            :  67.41%
  -> Random Forest  :  68.65%
  -> SVM            :  67.65%
  -> MLP            :  69.12%

Testing with  600 features...
  -> KNN            :  68.00%
  -> Random Forest  :  68.59%
  -> SVM            :  67.18%
  -> MLP            :  70.82%

Testing with  800 features...
  -> KNN            :  67.65%
  -> Random Forest  :  68.59%
  -> SVM            :  66.53%
  -> MLP            :  70.59%

Testing with 1024 features...
  -> KNN            :  67.47%
  -> Random Forest  :  68.24%
  -> SVM            :  66.82%
  -> MLP            :  70.47%


SUMMARY: BEST ACCURACY FOR EACH FEATURE COUNT

 Features    KN